# Combined RNA + Epigenetic Discordance Analysis
## Identifying TF/ncRNA-driven gene regulation in Layer V ET

**Discordance Score (WDS):**
```
WDS = α·Z(CGN) + β·Z(CHN) - Z(RNA)
```
- `WDS >> 0` → High methylation + Low expression → **FORCED_SUPPRESSION** (TF/ncRNA suppressing despite open-ish chromatin... wait, or blocking despite closed)
- `WDS << 0` → Low methylation + High expression → **FORCED_EXPRESSION** (TF/ncRNA forcing expression through closed chromatin)

**Narrowing logic:**
- Discordant in **all** cell types → not identity-specific
- Discordant only in **neurons** → neuron-identity regulation
- Discordant only in **deep layer neurons** → deep layer regulation
- Discordant only in **Layer V ET** → Layer V ET-specific regulation

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import zscore

sc.settings.verbosity = 1
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 50)

## 0. Configuration

In [ ]:
# ── EDIT THESE PATHS ─────────────────────────────────────────────────────────
PATH_RNA = "/home/nakagawa/datasets/h5ad/SMARTer_cells_MOp.h5ad"
PATH_CGN = "/home/nakagawa/datasets/h5ad/DNA_Methylation_CGN.h5ad"
PATH_CHN = "/home/nakagawa/datasets/h5ad/DNA_Methylation_CHN.h5ad"
OUT_DIR  = Path("/home/nakagawa/datasets/discordance_results")
# ─────────────────────────────────────────────────────────────────────────────

# Weighted Discordance Score weights (must sum to 1.0)
ALPHA = 0.6   # CGN weight (canonical CpG silencing)
BETA  = 0.4   # CHN weight (neuron-specific non-CpG silencing)

# Discordance thresholds
DS_THRESH = 2.0    # |WDS| >= this to call discordant (in Z-score units)

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Configuration set.")

## 1. Load All Three Datasets

In [ ]:
print("Loading RNA...")
adata_rna = sc.read_h5ad(PATH_RNA)
print(f"  RNA : {adata_rna.shape[0]} cells x {adata_rna.shape[1]} genes")

print("Loading CGN methylation...")
adata_cgn = sc.read_h5ad(PATH_CGN)
print(f"  CGN : {adata_cgn.shape[0]} cells x {adata_cgn.shape[1]} genes")

print("Loading CHN methylation...")
adata_chn = sc.read_h5ad(PATH_CHN)
print(f"  CHN : {adata_chn.shape[0]} cells x {adata_chn.shape[1]} genes")

## 2. Check Annotations Are Consistent Across Files

In [ ]:
print("=" * 60)
print("RNA metadata columns:")
print(adata_rna.obs.columns.tolist())

print("\nCGN metadata columns:")
print(adata_cgn.obs.columns.tolist())

print("\nCHN metadata columns:")
print(adata_chn.obs.columns.tolist())

In [ ]:
# ── EDIT: set the cell type column name for each file ────────────────────────
# Run the cell above first, then fill in the correct column names
CT_COL_RNA = "cell_type"   # e.g. 'cell_type', 'subclass_label', 'cluster'
CT_COL_CGN = "cell_type"   # may differ per file
CT_COL_CHN = "cell_type"
# ─────────────────────────────────────────────────────────────────────────────

ct_rna = set(adata_rna.obs[CT_COL_RNA].unique())
ct_cgn = set(adata_cgn.obs[CT_COL_CGN].unique())
ct_chn = set(adata_chn.obs[CT_COL_CHN].unique())

print(f"Cell types in RNA : {len(ct_rna)}")
print(f"Cell types in CGN : {len(ct_cgn)}")
print(f"Cell types in CHN : {len(ct_chn)}")

shared_ct = ct_rna & ct_cgn & ct_chn
print(f"\nShared across all three: {len(shared_ct)}")

only_rna = ct_rna - ct_cgn - ct_chn
only_cgn = ct_cgn - ct_rna - ct_chn
only_chn = ct_chn - ct_rna - ct_cgn

if only_rna: print(f"\n[WARNING] Only in RNA  : {only_rna}")
if only_cgn: print(f"[WARNING] Only in CGN  : {only_cgn}")
if only_chn: print(f"[WARNING] Only in CHN  : {only_chn}")

print("\nShared cell types:")
for ct in sorted(shared_ct):
    n_rna = (adata_rna.obs[CT_COL_RNA] == ct).sum()
    n_cgn = (adata_cgn.obs[CT_COL_CGN] == ct).sum()
    n_chn = (adata_chn.obs[CT_COL_CHN] == ct).sum()
    print(f"  {ct:<30} RNA={n_rna:>5}  CGN={n_cgn:>5}  CHN={n_chn:>5}")

In [ ]:
# Check gene name overlap
genes_rna = set(adata_rna.var_names)
genes_cgn = set(adata_cgn.var_names)
genes_chn = set(adata_chn.var_names)

shared_genes = genes_rna & genes_cgn & genes_chn
print(f"Genes in RNA       : {len(genes_rna)}")
print(f"Genes in CGN       : {len(genes_cgn)}")
print(f"Genes in CHN       : {len(genes_chn)}")
print(f"Shared across all  : {len(shared_genes)}")

# Sample check — first 10 gene names per file
print("\nRNA gene name examples :", list(adata_rna.var_names[:5]))
print("CGN gene name examples :", list(adata_cgn.var_names[:5]))
print("CHN gene name examples :", list(adata_chn.var_names[:5]))

## 3. Define Cell Type Groups
Fill in after confirming shared cell types above.

In [ ]:
# ── EDIT: use exact strings from the shared cell type list above ──────────────
LAYER_V_ET = "L5 ET"

NON_NEURON_TYPES = [
    "Astrocyte",
    "Microglia",
    "Oligodendrocyte",
    "OPC",
    "Endothelial",
    "VLMC",
]

UPPER_LAYER_TYPES = [
    "L2/3 IT",
    "L4 IT",
]

OTHER_DEEP_LAYER_TYPES = [
    "L5 IT",
    "L5/6 NP",
    "L6 IT",
    "L6 CT",
    "L6b",
]
# ─────────────────────────────────────────────────────────────────────────────

ALL_CELL_TYPES = [LAYER_V_ET] + NON_NEURON_TYPES + UPPER_LAYER_TYPES + OTHER_DEEP_LAYER_TYPES

# Filter to only those actually present in all three datasets
ALL_CELL_TYPES = [ct for ct in ALL_CELL_TYPES if ct in shared_ct]
print(f"Working with {len(ALL_CELL_TYPES)} cell types present in all three datasets:")
for ct in ALL_CELL_TYPES:
    print(f"  {ct}")

## 4. Compute Pseudobulk Mean Per Cell Type
We average across cells per cell type — this gives one value per gene per cell type,
which is what we Z-score across cell types.

In [ ]:
def pseudobulk_mean(adata, celltype_col, cell_types, gene_set):
    """
    Returns DataFrame: rows=genes (shared_genes), cols=cell_types
    Values = mean expression/methylation per cell type.
    """
    gene_list = sorted(gene_set & set(adata.var_names))
    result = {}
    for ct in cell_types:
        mask = adata.obs[celltype_col] == ct
        sub  = adata[mask][:, gene_list]
        if hasattr(sub.X, 'toarray'):
            vals = sub.X.toarray()
        else:
            vals = np.array(sub.X)
        result[ct] = vals.mean(axis=0)
    return pd.DataFrame(result, index=gene_list)   # genes x cell_types


print("Computing pseudobulk means...")

# RNA: normalize first if raw counts
adata_rna_use = adata_rna.raw.to_adata() if adata_rna.raw else adata_rna.copy()
adata_rna_use.obs[CT_COL_RNA] = adata_rna.obs[CT_COL_RNA]
if adata_rna_use.X.max() > 100:
    sc.pp.normalize_total(adata_rna_use, target_sum=1e4)
    sc.pp.log1p(adata_rna_use)
    print("  RNA: normalized + log1p")

pb_rna = pseudobulk_mean(adata_rna_use, CT_COL_RNA, ALL_CELL_TYPES, shared_genes)
print(f"  RNA pseudobulk : {pb_rna.shape}")

# CGN/CHN: methylation fractions, no normalization needed
pb_cgn = pseudobulk_mean(adata_cgn, CT_COL_CGN, ALL_CELL_TYPES, shared_genes)
print(f"  CGN pseudobulk : {pb_cgn.shape}")

pb_chn = pseudobulk_mean(adata_chn, CT_COL_CHN, ALL_CELL_TYPES, shared_genes)
print(f"  CHN pseudobulk : {pb_chn.shape}")

# Align gene index
common_idx = pb_rna.index.intersection(pb_cgn.index).intersection(pb_chn.index)
pb_rna = pb_rna.loc[common_idx]
pb_cgn = pb_cgn.loc[common_idx]
pb_chn = pb_chn.loc[common_idx]
print(f"\nFinal gene set for analysis: {len(common_idx)} genes")

## 5. Z-score Normalize Each Matrix
Z-scored **across cell types** per gene — captures relative deviation from the mean cell type behaviour.

In [ ]:
def zscore_across_celltypes(df):
    """
    Z-score each gene (row) across cell types (columns).
    Genes with zero variance across cell types get DS=0 (they're not informative).
    """
    arr    = df.values.astype(float)
    means  = arr.mean(axis=1, keepdims=True)
    stds   = arr.std(axis=1, keepdims=True)
    stds[stds == 0] = 1   # avoid division by zero
    z      = (arr - means) / stds
    return pd.DataFrame(z, index=df.index, columns=df.columns)


z_rna = zscore_across_celltypes(pb_rna)
z_cgn = zscore_across_celltypes(pb_cgn)
z_chn = zscore_across_celltypes(pb_chn)

print("Z-score ranges (should be centred near 0):")
print(f"  RNA : min={z_rna.values.min():.2f}  max={z_rna.values.max():.2f}")
print(f"  CGN : min={z_cgn.values.min():.2f}  max={z_cgn.values.max():.2f}")
print(f"  CHN : min={z_chn.values.min():.2f}  max={z_chn.values.max():.2f}")

## 6. Compute Weighted Discordance Score (WDS)
```
WDS = α·Z(CGN) + β·Z(CHN) - Z(RNA)
WDS >> 0 → FORCED_SUPPRESSION  (high methylation, low RNA)
WDS << 0 → FORCED_EXPRESSION   (low methylation, high RNA)
```

In [ ]:
wds = ALPHA * z_cgn + BETA * z_chn - z_rna   # genes x cell_types

print(f"WDS matrix shape : {wds.shape}  (genes x cell types)")
print(f"WDS range        : {wds.values.min():.2f} to {wds.values.max():.2f}")
print(f"Discordance threshold: |WDS| >= {DS_THRESH}")
print(f"\nGenes with |WDS| >= {DS_THRESH} in Layer V ET ({LAYER_V_ET}):")
n_forced_on  = (wds[LAYER_V_ET] <= -DS_THRESH).sum()
n_forced_off = (wds[LAYER_V_ET] >=  DS_THRESH).sum()
print(f"  FORCED_EXPRESSION  (WDS <= -{DS_THRESH}): {n_forced_on} genes")
print(f"  FORCED_SUPPRESSION (WDS >=  {DS_THRESH}): {n_forced_off} genes")

## 7. Build Per-Cell-Type Discordance Tables

In [ ]:
def get_discordant_genes(wds_col, z_rna_col, z_cgn_col, z_chn_col, threshold, celltype_name):
    """
    For one cell type, return a DataFrame of discordant genes with all scores.
    """
    df = pd.DataFrame({
        "gene"       : wds_col.index,
        "WDS"        : wds_col.values,
        "Z_RNA"      : z_rna_col.values,
        "Z_CGN"      : z_cgn_col.values,
        "Z_CHN"      : z_chn_col.values,
        "cell_type"  : celltype_name,
    })
    df["category"] = "CONCORDANT"
    df.loc[df["WDS"] <= -threshold, "category"] = "FORCED_EXPRESSION"
    df.loc[df["WDS"] >=  threshold, "category"] = "FORCED_SUPPRESSION"
    df = df[df["category"] != "CONCORDANT"].copy()
    df = df.sort_values("WDS", key=abs, ascending=False).reset_index(drop=True)
    return df


discordant_per_ct = {}   # cell_type -> DataFrame of discordant genes

for ct in ALL_CELL_TYPES:
    df = get_discordant_genes(
        wds[ct], z_rna[ct], z_cgn[ct], z_chn[ct], DS_THRESH, ct
    )
    discordant_per_ct[ct] = df
    n_fe = (df["category"] == "FORCED_EXPRESSION").sum()
    n_fs = (df["category"] == "FORCED_SUPPRESSION").sum()
    print(f"  {ct:<30} FORCED_EXPRESSION={n_fe:>4}  FORCED_SUPPRESSION={n_fs:>4}")

# Save individual CSVs
print("\nSaving per-cell-type CSVs...")
for ct, df in discordant_per_ct.items():
    safe = ct.replace("/", "_").replace(" ", "_")
    path = OUT_DIR / f"discordant_{safe}.csv"
    df.to_csv(path, index=False)
print(f"Saved {len(discordant_per_ct)} files to {OUT_DIR}/")

## 8. Stepwise Narrowing to Layer V ET-Specific Discordance

For each direction (FORCED_EXPRESSION / FORCED_SUPPRESSION) separately:
- **All cell types** share it → not identity-specific
- **Only neurons** share it → neuron-identity regulation
- **Only deep layer neurons** share it → deep layer regulation
- **Only Layer V ET** → Layer V ET-specific regulation

In [ ]:
def gene_set(ct, category):
    """Return set of genes for a given cell type and discordance category."""
    if ct not in discordant_per_ct:
        return set()
    df = discordant_per_ct[ct]
    return set(df[df["category"] == category]["gene"])


def union_sets(cell_types, category):
    sets = [gene_set(ct, category) for ct in cell_types if ct in discordant_per_ct]
    return set.union(*sets) if sets else set()


def intersect_sets(cell_types, category):
    sets = [gene_set(ct, category) for ct in cell_types if ct in discordant_per_ct]
    return set.intersection(*sets) if sets else set()


results = {}

for cat in ["FORCED_EXPRESSION", "FORCED_SUPPRESSION"]:
    print(f"\n{'='*60}")
    print(f"Category: {cat}")
    print('='*60)

    layerVET_genes    = gene_set(LAYER_V_ET, cat)
    non_neuron_genes  = union_sets(NON_NEURON_TYPES,       cat)
    upper_layer_genes = union_sets(UPPER_LAYER_TYPES,      cat)
    deep_layer_genes  = union_sets(OTHER_DEEP_LAYER_TYPES, cat)
    all_neuron_genes  = upper_layer_genes | deep_layer_genes | layerVET_genes

    # Tier 1: present in ALL cell types (non-specific)
    universal = layerVET_genes & non_neuron_genes

    # Tier 2: neuron-specific (in neurons but NOT in non-neurons)
    neuron_specific = (layerVET_genes & all_neuron_genes) - non_neuron_genes

    # Tier 3: deep layer specific (in deep layer neurons but NOT upper layer or non-neuron)
    deep_specific = (layerVET_genes & deep_layer_genes) - upper_layer_genes - non_neuron_genes

    # Tier 4: Layer V ET specific (ONLY in Layer V ET, not in any other type)
    all_other_genes = non_neuron_genes | upper_layer_genes | deep_layer_genes
    layerVET_specific = layerVET_genes - all_other_genes

    print(f"  Layer V ET discordant genes total : {len(layerVET_genes)}")
    print(f"  Tier 0 — Universal (non-specific) : {len(universal)}")
    print(f"  Tier 1 — Neuron-identity specific  : {len(neuron_specific)}")
    print(f"  Tier 2 — Deep layer specific       : {len(deep_specific)}")
    print(f"  Tier 3 — Layer V ET specific       : {len(layerVET_specific)}")

    results[cat] = {
        "universal"       : universal,
        "neuron_specific" : neuron_specific,
        "deep_specific"   : deep_specific,
        "layerVET_specific": layerVET_specific,
    }

## 9. Build Summary DataFrames with Full WDS Scores

In [ ]:
def build_wds_summary(gene_set_input, tier_label, category):
    """
    For each gene, collect WDS, Z_RNA, Z_CGN, Z_CHN across all cell types.
    """
    if not gene_set_input:
        return pd.DataFrame()

    rows = []
    for gene in sorted(gene_set_input):
        row = {"gene": gene, "tier": tier_label, "category": category}
        for ct in ALL_CELL_TYPES:
            if gene in wds.index:
                row[f"{ct}__WDS"]   = round(wds.loc[gene, ct], 3)
                row[f"{ct}__Z_RNA"] = round(z_rna.loc[gene, ct], 3)
                row[f"{ct}__Z_CGN"] = round(z_cgn.loc[gene, ct], 3)
                row[f"{ct}__Z_CHN"] = round(z_chn.loc[gene, ct], 3)
        rows.append(row)

    df = pd.DataFrame(rows)
    # Add mean WDS across all cell types for sorting
    wds_cols = [c for c in df.columns if c.endswith("__WDS")]
    df["mean_WDS_all_ct"] = df[wds_cols].mean(axis=1).round(3)
    df = df.sort_values("mean_WDS_all_ct", key=abs, ascending=False).reset_index(drop=True)
    return df


tier_labels = {
    "universal"        : "Tier0_Universal",
    "neuron_specific"  : "Tier1_Neuron_Specific",
    "deep_specific"    : "Tier2_DeepLayer_Specific",
    "layerVET_specific": "Tier3_LayerVET_Specific",
}

all_summaries = {}

for cat in ["FORCED_EXPRESSION", "FORCED_SUPPRESSION"]:
    for tier_key, tier_label in tier_labels.items():
        gene_s = results[cat][tier_key]
        df     = build_wds_summary(gene_s, tier_label, cat)
        key    = f"{cat}__{tier_label}"
        all_summaries[key] = df

print("Summary DataFrames built.")
for k, v in all_summaries.items():
    print(f"  {k}: {len(v)} genes")

## 10. Save All Results to CSV

In [ ]:
# Save individual tier/category files
for key, df in all_summaries.items():
    if df.empty:
        print(f"[SKIP] {key} — empty")
        continue
    path = OUT_DIR / f"{key}.csv"
    df.to_csv(path, index=False)
    print(f"Saved: {path.name}  ({len(df)} genes)")

# Save one master file combining everything
master = pd.concat([df for df in all_summaries.values() if not df.empty], ignore_index=True)
master_path = OUT_DIR / "MASTER_discordance_summary.csv"
master.to_csv(master_path, index=False)
print(f"\nMaster file saved: {master_path.name}  ({len(master)} genes total)")

## 11. Preview: Layer V ET-Specific Genes

In [ ]:
preview_cols = ["gene", "category", "tier", "mean_WDS_all_ct",
                f"{LAYER_V_ET}__WDS", f"{LAYER_V_ET}__Z_RNA",
                f"{LAYER_V_ET}__Z_CGN", f"{LAYER_V_ET}__Z_CHN"]

for cat in ["FORCED_EXPRESSION", "FORCED_SUPPRESSION"]:
    key = f"{cat}__Tier3_LayerVET_Specific"
    df  = all_summaries.get(key, pd.DataFrame())
    print(f"\n{'='*60}")
    print(f"Layer V ET-Specific {cat} (top 20)")
    print('='*60)
    if df.empty:
        print("No genes found — consider lowering DS_THRESH in Cell 0")
    else:
        available = [c for c in preview_cols if c in df.columns]
        display(df[available].head(20))

## 12. Threshold Sensitivity Check

In [ ]:
print(f"{'DS_thresh':>10}  {'FE_Tier3':>10}  {'FS_Tier3':>10}  {'FE_Tier1':>10}  {'FS_Tier1':>10}")
print("-" * 55)

for thresh in [1.0, 1.5, 2.0, 2.5, 3.0]:
    tmp = {}
    for ct in ALL_CELL_TYPES:
        for cat in ["FORCED_EXPRESSION", "FORCED_SUPPRESSION"]:
            col = wds[ct]
            if cat == "FORCED_EXPRESSION":
                genes = set(col[col <= -thresh].index)
            else:
                genes = set(col[col >=  thresh].index)
            tmp[(ct, cat)] = genes

    def gs(ct, cat): return tmp.get((ct, cat), set())
    def u(cts, cat): return set.union(*[gs(ct, cat) for ct in cts]) if cts else set()

    for cat, sign in [("FORCED_EXPRESSION", "FE"), ("FORCED_SUPPRESSION", "FS")]:
        lv   = gs(LAYER_V_ET, cat)
        non  = u(NON_NEURON_TYPES, cat)
        upl  = u(UPPER_LAYER_TYPES, cat)
        dep  = u(OTHER_DEEP_LAYER_TYPES, cat)
        tmp[f"{sign}_t3_{thresh}"] = len(lv - non - upl - dep)
        tmp[f"{sign}_t1_{thresh}"] = len((lv | upl | dep) - non)

    print(f"{thresh:>10.1f}  "
          f"{tmp['FE_t3_' + str(thresh)]:>10}  "
          f"{tmp['FS_t3_' + str(thresh)]:>10}  "
          f"{tmp['FE_t1_' + str(thresh)]:>10}  "
          f"{tmp['FS_t1_' + str(thresh)]:>10}")

print("\nFE=FORCED_EXPRESSION, FS=FORCED_SUPPRESSION, Tier3=LayerVET-specific, Tier1=Neuron-specific")

## 13. Intersect with RNA DE Results (Optional)
Cross-reference with Layer V ET-specific genes from the RNA notebook for highest-confidence candidates.

In [ ]:
RNA_DE_PATH = "/home/nakagawa/datasets/LayerV_ET_results/LayerVET_specific_genes.csv"

try:
    rna_de = pd.read_csv(RNA_DE_PATH)
    rna_de_genes = set(rna_de["gene"])
    print(f"Loaded {len(rna_de_genes)} Layer V ET-specific genes from RNA DE analysis")

    for cat in ["FORCED_EXPRESSION", "FORCED_SUPPRESSION"]:
        key = f"{cat}__Tier3_LayerVET_Specific"
        df  = all_summaries.get(key, pd.DataFrame())
        if df.empty:
            continue
        overlap = set(df["gene"]) & rna_de_genes
        print(f"\n{cat} Tier3 x RNA DE overlap: {len(overlap)} genes")
        print("  → These are your TOP PRIORITY candidates (discordant + DE specific)")
        for g in sorted(overlap):
            wds_val = wds.loc[g, LAYER_V_ET] if g in wds.index else float('nan')
            print(f"    {g:<20} WDS={wds_val:.3f}")

        # Save overlap
        overlap_df = df[df["gene"].isin(overlap)].copy()
        overlap_df.to_csv(OUT_DIR / f"TOP_PRIORITY_{cat}.csv", index=False)

except FileNotFoundError:
    print("RNA DE results not found — run RNA notebook first, or update RNA_DE_PATH above")